# 03 - Machine Learning Models
Training Logistic Regression, Random Forest, and XGBoost with SMOTE.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib
import os


In [ ]:
df = pd.read_csv('../data/processed/stroke_processed.csv')
X = df.drop(['id', 'stroke'], axis=1)
y = df['stroke']


## Train-Test Split & Scaling


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## SMOTE for Class Imbalance


In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)


## XGBoost Training


In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
model.fit(X_train_sm, y_train_sm)


In [ ]:
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:,1]))


## Save Best Model & Scaler


In [ ]:
os.makedirs('../models/saved', exist_ok=True)
joblib.dump(model, '../models/saved/best_ml_model.pkl')
joblib.dump(scaler, '../models/saved/scaler.pkl')
